# Data exploration — the datasets we EVALUATE on

These 21 datasets are what every model is **scored** on. They are not training data and
they are not priors: this notebook is about the measurement instrument, not the method.

Three questions it answers:

| question | why it decides something |
|---|---|
| **How much boundary mass does each LGD target have?** | Where it is large, a model that only predicts smooth interior values cannot be calibrated — it can get the average right while being wrong about every individual loan. |
| **How rare is default in each PD dataset?** | Sets the imbalance the models must handle, and shows how far real data sits from a balanced prior. |
| **What shape, types and missingness?** | Sets the ranges the prior samples over, and flags anything that would undermine a published number. |

Everything reads the **processed parquet cache** through `src.data.pipeline`, so this
notebook sees exactly the tables the evaluation sees — not a separate re-read of the raw
CSVs that might disagree. Anything not yet processed is processed on first access, which
takes a few minutes the first time.

All plots live in `src/visualize/data_plots.py`. **Ends with a copy-pasteable text
summary.**

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src.visualize import data_plots, figures, style, summaries

style.apply()   # ONE shared style: identical colours in every notebook
pd.set_option("display.width", 220, "display.max_columns", 60, "display.max_rows", 60)

# Clears THIS notebook's figure folder — and no other — BEFORE anything is
# drawn, then saves every figure as PDF (print) + PNG (committable). Identical
# behaviour in Jupyter and under src/utils/run_notebooks.py.
FIGS = figures.FigureSaver("data_exploration")

## 1. Load everything

A dataset that cannot be read is **skipped with a message** rather than killing the
notebook — one bad raw file should not stop you looking at the other twenty.

In [ ]:
lgd = data_plots.load_all("lgd")
pd_ = data_plots.load_all("pd")
both = {"lgd": lgd, "pd": pd_}
print(f"\nloaded {len(lgd)} LGD and {len(pd_)} PD datasets")

## 2. LGD — the boundary-mass story

Read **`boundary mass`** and **`in [0,1]`** first. `boundary mass` is the fraction of rows
sitting exactly at the minimum or maximum.

In [ ]:
data_plots.summary_table(lgd, 'lgd')

### 2a. Every LGD target, one panel each

The motivating figure of the project. Compare the two end bars against the middle.
Dashed red lines mark exact atoms — in a 40-bin histogram, an atom at 0 and a cluster at
0.02 look identical, and only one of them is the point.

In [ ]:
FIGS.save(
    data_plots.plot_lgd_targets(lgd),
    "lgd_targets",
    caption=(
        "Histograms of the Loss Given Default target for each of the seven LGD datasets, ordered "
        "by sample size. Forty bins per panel. Dashed vertical lines mark the minimum and maximum "
        "observed values where more than 1% of observations fall exactly on them. Panel subtitles "
        "give the combined share of observations at the two boundaries."
    ),
);


### 2b. Which datasets does it actually matter for?

A ranked list, because that is the practical question. Two things to take away: the
spread is **wide** (so the prior needs a range, not a value), and mass at 0 and at 1 are
**not symmetric** (so the prior samples them separately).

In [ ]:
FIGS.save(
    data_plots.plot_boundary_mass_ranking(lgd),
    "boundary_mass_ranking",
    caption=(
        "Share of observations lying exactly at a boundary of the observed target range, per LGD "
        "dataset, ordered by total. Bars are split into mass at the minimum (blue) and at the "
        "maximum (orange). Percentages give the total per dataset."
    ),
);


## 3. PD — the imbalance story

`imbalance 1:n` restates the base rate as odds, which is easier to feel: 1:99 means one
default per hundred loans.

In [ ]:
data_plots.summary_table(pd_, 'pd')

In [ ]:
FIGS.save(
    data_plots.plot_pd_base_rates(pd_),
    "pd_base_rates",
    caption=(
        "Default rate per PD dataset, ordered by rate, on a logarithmic horizontal axis. The "
        "dashed vertical line marks a 50% rate. Percentages give the rate per dataset."
    ),
);


Log axis, because the rates span two orders of magnitude; on a linear axis every dataset
below 5% collapses into the same sliver.

The dashed grey line is roughly where TabICL's prior sits. **Every real dataset is to the
left of it.**

## 4. Shape, types and missingness

If the prior generated 500-column tables and every real dataset had 20, the extra
capacity would be wasted; if it generated 200-row tables and real ones have a million, it
would never learn to use a long context.

In [ ]:
FIGS.save(
    data_plots.plot_shapes(both),
    "shapes",
    caption=(
        "Number of rows against number of features for all 21 evaluation datasets, both axes "
        "logarithmic. Colour denotes task; each point is labelled with its dataset name."
    ),
);


In [ ]:
FIGS.save(
    data_plots.plot_type_mix(both),
    "type_mix",
    caption=(
        "Share of columns that are categorical, per dataset, ordered by share. Colour denotes "
        "task."
    ),
);


In [ ]:
FIGS.save(
    data_plots.plot_missingness(both),
    "missingness",
    caption=(
        "Share of cells that are missing, per dataset, ordered by share, measured after "
        "preprocessing. Colour denotes task."
    ),
);


Note how many datasets have **zero** missing values: most were imputed before we received
them, so real missingness is understated here. Our prior still injects missingness as an
explicit mechanism — a model that has never seen a NaN handles one badly — but these
numbers measure the upstream pipeline, not the domain, so do not tune to them.

## 5. Feature dependence

Real credit data comes in **blocks** of strongly correlated columns: several measures of
the same balance, several vintages of the same delinquency count. Our prior builds
features through random DAGs precisely so those blocks appear. If these heatmaps were
diagonal, that design choice would be wrong.

In [ ]:
FIGS.save(
    data_plots.plot_feature_correlations(lgd, n_show=7),
    "feature_correlations_lgd",
    caption=(
        "Pearson correlation matrices between features for the six largest datasets of the task, "
        "computed on the first 5,000 rows with constant columns removed. Colour scale spans -1 to "
        "1. Panel subtitles give the number of columns retained and the mean absolute off- "
        "diagonal correlation."
    ),
);


In [ ]:
FIGS.save(
    data_plots.plot_feature_correlations(pd_, n_show=15),
    "feature_correlations_pd",
    caption=(
        "Pearson correlation matrices between features for the six largest datasets of the task, "
        "computed on the first 5,000 rows with constant columns removed. Colour scale spans -1 to "
        "1. Panel subtitles give the number of columns retained and the mean absolute off- "
        "diagonal correlation."
    ),
);


## 6. Leakage screen

For every column, its absolute correlation with the target. This exists because
`lgd_lendingclub` gives R² around 0.71–0.76 — far above anything reported for LGD
modelling, which usually means a column encodes the answer.

A high correlation is a **pointer, not proof**: a single strong predictor can be
legitimate. And this only looks at one feature at a time, so it cannot see leakage spread
across several columns.

In [ ]:
leak_lgd = data_plots.leakage_check(lgd, "lgd")
leak_lgd.head(12)

In [ ]:
leak_pd = data_plots.leakage_check(pd_, "pd")
leak_pd.head(12)

In [ ]:
flagged = pd.concat([leak_lgd, leak_pd])
flagged[flagged["suspicious"]]

---

## 7. TEXT SUMMARY

Everything above, as text. **Copy-paste this** — it carries the numbers.

In [ ]:
print(summaries.data_summary(both, leakage=pd.concat([leak_lgd, leak_pd])))
print()
print(FIGS.summary())